In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
import IPython

pd.set_option('display.max_columns', None)

# Data Loading

In [ ]:
DATA_PATH = "results/MAG2P_order_parameters-2025-12-8-16:13:13.pickle"

dg = pd.read_pickle(DATA_PATH)
df = dg.fillna(0)
df = df.drop(columns=["std_bonds_1_8", "std_bonds_1_5", "std_size",
                       "std_radius_of_gyration", "std_second_neighbours"])
df = df.loc[:, (df != 0).any(axis=0)]

print(f"Loaded {len(df)} rows, {len(df.columns)} columns")
df.head()

# Feature Groups & Test Sets

In [ ]:
# Define named feature groups by column index ranges
# Columns 0-2: file_id, lambda, shift (not features)
# Columns 3-8: global structural measures
# Columns 9-32: orientation (24 angular bins)
# Columns 33-61: radius of gyration (29 bins)
# Columns 62-86: g(r) (25 radial distance bins)

FEATURE_GROUPS = {
    "global":      df.columns[3:9],
    "orientation": df.columns[9:33],
    "Rg":          df.columns[33:62],
    "gofr":        df.columns[62:],
}

# Define test sets: name -> list of groups to KEEP
TEST_SETS = {
    "all_features":        list(FEATURE_GROUPS.keys()),
    "no_functions":        ["global"],
    "no_orientation_Rg":   ["global", "gofr"],
    "no_orientation_gofr": ["global", "Rg"],
    "no_Rg_gofr":          ["global", "orientation"],
    "no_global_Rg":        ["orientation", "gofr"],
    "no_orientation":      ["global", "Rg", "gofr"],
    "no_Rg":               ["global", "orientation", "gofr"],
}

# Print summary
for name, groups in TEST_SETS.items():
    n_cols = sum(len(FEATURE_GROUPS[g]) for g in groups)
    print(f"{name:25s} -> {groups}  ({n_cols} features)")

# Diffusion Map Analysis

In [ ]:
import hdbscan
import seaborn as sns
import cmasher as cmr
from sklearn.manifold import SpectralEmbedding
from matplotlib.colors import ListedColormap, rgb2hex


def run_diffusion_heatmap(X, lambdas, shifts, set_name, cluster_min_size=10):
    """
    Perform Diffusion Maps on data X, assign clusters with HDBSCAN,
    and create heatmaps over (shift, lambda) space.
    Outputs are saved with set_name suffix.
    """
    embedding = SpectralEmbedding(
        n_components=2, affinity='nearest_neighbors', n_neighbors=32
    )
    diffmap_emb = embedding.fit_transform(X)
    cluster_labels = hdbscan.HDBSCAN(
        min_cluster_size=cluster_min_size, core_dist_n_jobs=-1
    ).fit_predict(diffmap_emb)

    df_cluster = pd.DataFrame({
        'lambda': lambdas,
        'shift': shifts,
        'cluster': cluster_labels,
        'Diff-1': diffmap_emb[:, 0],
        'Diff-2': diffmap_emb[:, 1],
    })

    # Color map: distinct colors for clusters, gray for noise
    unique_clusters = sorted(df_cluster['cluster'].unique())
    non_noise = [cl for cl in unique_clusters if cl != -1]
    palette = cmr.chroma(np.linspace(0, 1, max(len(non_noise), 1)))
    hex_colors = [rgb2hex(c) for c in palette]
    color_map = {str(cl): hex_colors[i] for i, cl in enumerate(non_noise)}
    color_map['-1'] = '#A0A0A0'

    # Diffusion Map scatter
    fig = px.scatter(
        df_cluster, x='Diff-1', y='Diff-2',
        color=df_cluster['cluster'].astype(str),
        hover_data={'lambda': True, 'shift': True},
        title=f'Diffusion Map - {set_name}',
        color_discrete_map=color_map,
    )
    fig.write_html(f'diffmap_clusters_{set_name}.html')
    fig.show()

    # Discrete cluster heatmap
    heatmap_data = df_cluster.groupby(['lambda', 'shift'])['cluster'] \
                             .agg(lambda x: x.value_counts().idxmax()) \
                             .unstack().sort_index(ascending=False)

    unique_hm = sorted(heatmap_data.stack().dropna().unique())
    non_noise_hm = [cl for cl in unique_hm if cl != -1]
    colors = cmr.chroma(np.linspace(0, 1, max(len(non_noise_hm), 1)))
    colors_hex = [rgb2hex(c) for c in colors]
    colors_all = ['#A0A0A0' if cl == -1 else colors_hex.pop(0) for cl in unique_hm]
    cmap = ListedColormap(colors_all)

    plt.figure(figsize=(10, 8))
    sns.heatmap(heatmap_data, cmap=cmap, annot=False, cbar=True,
                linewidths=0.5, linecolor='gray')
    plt.title(f'HDBSCAN Clusters - {set_name}')
    plt.xlabel('shift')
    plt.ylabel('lambda')
    plt.tight_layout()
    plt.savefig(f'heatmap_diffmap_{set_name}.pdf')
    plt.show()

    # Continuous Diff-1 heatmap
    cont_data = df_cluster.groupby(['lambda', 'shift'])['Diff-1'] \
                          .mean().unstack().sort_index(ascending=False)

    plt.figure(figsize=(10, 8))
    sns.heatmap(cont_data, cmap=cmr.chroma, annot=False, cbar=True,
                linewidths=0.5, linecolor='gray')
    plt.title(f'Average Diff-1 - {set_name}')
    plt.xlabel('shift')
    plt.ylabel('lambda')
    plt.tight_layout()
    plt.savefig(f'heatmap_diffmap_continuous_{set_name}.pdf')
    plt.show()

    return df_cluster

# Run All Test Sets

In [ ]:
from sklearn.preprocessing import StandardScaler

Shift = df['shift']
Lambda = df['lambda']

results = {}
for name, keep_groups in TEST_SETS.items():
    print(f'\n{"="*60}')
    print(f'Running: {name} (groups: {keep_groups})')
    print(f'{"="*60}')

    feature_cols = [c for g in keep_groups for c in FEATURE_GROUPS[g]]
    X = df[feature_cols].values
    X = StandardScaler().fit_transform(X)

    results[name] = run_diffusion_heatmap(X, Lambda, Shift, set_name=name, cluster_min_size=5)